# SNCP-PPO Social Navigation - Colab Notebook

End-to-end notebook for preparing and running the current **SNCP-PPO** experiment on Colab.

## Run order
1. **Setup** - clone repo, install deps, optional Drive mount
2. **Smoke test** - verify env + model + a tiny training loop
3. **V38 action-shield probe** - quick GO check, then wide eval if quick passes
4. **Analyze** - read the generated GO/NO-GO report
5. **Visualize / inspect** - optional generated reports
6. **Persist** - download probe artifacts before the session ends

## Current run: v38 - training-free action shield probe

v37 is complete and `NO-GO`; full v37 training is cancelled. The locked champion/base remains
`v34-fixed-beta`, supplied as `sncp_ppo_v34.pt` at the repo root. V38 performs **no PPO training**.
It evaluates the same checkpoint twice on the same episode bank:

- **C0 control:** raw deterministic v34 policy action.
- **C1 shield:** v34 action post-processed by a short-horizon collision-risk action shield.
- Quick probe already passed once: N=15/20, 50 episodes per arm/density, output `eval_v38_shield_probe/`.
- Current next step: wide eval `5 10 15 20` / `100` episodes, output `eval_v38_shield_full/`.

## Colab tips
- Runtime GPU is optional; V38 is eval-only, not training.
- Upload or copy `sncp_ppo_v34.pt` into the repo root before Section 3.
- Mount Drive (Section 1.4) so artifacts survive a disconnect.


## 1. Setup

### 1.1 GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

### 1.2 Clone / update repository

Re-run after any push to pull the latest code. Note: this updates the repo files on the VM, **not** an already-open notebook — reopen the notebook from GitHub to get notebook changes.

In [ ]:
import os
REPO_URL = 'https://github.com/heimdilon/sncp-ppo-crowdnav.git'
REPO_DIR = '/content/sncp-ppo-crowdnav'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned. Pulling latest...')
    !cd {REPO_DIR} && git pull --rebase

%cd {REPO_DIR}
!git log --oneline -1

### 1.3 Install dependencies

In [ ]:
!pip install -q -r requirements.txt

import torch
print(f'torch     {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'          device: {torch.cuda.get_device_name(0)}')

import gymnasium, ncps, numpy, matplotlib
print(f'gymnasium {gymnasium.__version__}')
print(f'ncps      {ncps.__version__}')
print(f'numpy     {numpy.__version__}')

### 1.4 (Optional) Mount Google Drive

Set `USE_DRIVE = True` to persist `checkpoints/` and `logs/` across sessions (recommended for long runs — a disconnect mid-training otherwise loses everything).

In [ ]:
USE_DRIVE = False  # set True to persist runs across Colab sessions
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/sncp-ppo-crowdnav-runs'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(f'{DRIVE_PROJECT_DIR}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_PROJECT_DIR}/logs', exist_ok=True)
    import shutil
    for sub in ('checkpoints', 'logs'):
        local = f'{REPO_DIR}/{sub}'
        if os.path.islink(local):
            os.unlink(local)
        elif os.path.isdir(local):
            for f in os.listdir(local):
                dst = f'{DRIVE_PROJECT_DIR}/{sub}/{f}'
                if not os.path.exists(dst):
                    shutil.copy2(f'{local}/{f}', dst)
            shutil.rmtree(local)
        os.symlink(f'{DRIVE_PROJECT_DIR}/{sub}', local)
    print(f'Drive-backed dirs: {DRIVE_PROJECT_DIR}/{{checkpoints,logs}}')
else:
    print('Drive mount skipped (USE_DRIVE=False). Files are lost when the Colab session ends.')

## 2. Smoke tests

Fast sanity checks before spending GPU hours. Env + model first, then a 50-episode single-env training loop (legacy path) that exercises curriculum, holdout, value clipping, LR schedule, and the per-update diagnostics line.

In [ ]:
!python test_env.py

In [ ]:
!python test_model.py

In [ ]:
# 50-episode single-env smoke. NOT the full run (that's Section 3). Replay is 0
# here so this stays a quick baseline check of the single-env path.
import subprocess, sys
cmd = [
    sys.executable, '-u', '-m', 'sncp_ppo.train',
    '--episodes', '50',
    '--num_humans', '5',
    '--seed', '42',
    '--eval_freq', '25',
    '--holdout_episodes', '3',
    '--holdout_scenarios', 'easy', 'hard',
    '--update_freq', '5',
    '--log_freq', '10',
    '--curriculum_replay_ratio', '0.0',
    '--save_path', 'checkpoints/sncp_ppo_smoke.pt',
]
print('Running:', ' '.join(cmd))
print('=' * 80)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()
print(f'\nExited with code {p.returncode}')

## 3. V38 action-shield eval (no training)

This section runs `scripts/run_v38_shield_probe.py`:

| Arm | Meaning |
| --- | --- |
| C0 | raw deterministic v34 policy |
| C1 | v34 policy action filtered by the v38 runtime action shield |

The shield only intervenes when its short constant-velocity rollout predicts a collision. Default
`safety_margin=0.0` is intentionally conservative; an earlier `0.10m` buffer caused unnecessary
interventions in a local smoke.


### 3.1 Run quick shield probe

Before running this cell, make sure `sncp_ppo_v34.pt` exists at the repo root. If it is in Drive, copy it here first.


In [ ]:
# v38 = training-free action shield probe over the locked v34-fixed-beta checkpoint.
BASE_CHECKPOINT = 'sncp_ppo_v34.pt'
OUTPUT_DIR = 'eval_v38_shield_probe'
DENSITIES = [15, 20]
EVAL_EPISODES = 50
SHIELD_HORIZON_STEPS = 6
SHIELD_SAFETY_MARGIN = 0.0

import os, subprocess, sys

if not os.path.exists(BASE_CHECKPOINT):
    raise FileNotFoundError(
        f'{BASE_CHECKPOINT} not found. Upload/copy the locked v34-fixed-beta checkpoint '
        'to the repo root before running the v38 shield probe.'
    )

cmd = [
    sys.executable, '-u', 'scripts/run_v38_shield_probe.py',
    '--checkpoint', BASE_CHECKPOINT,
    '--output_dir', OUTPUT_DIR,
    '--densities', *[str(n) for n in DENSITIES],
    '--n_episodes', str(EVAL_EPISODES),
    '--seed', '100',
    '--robot_vpref', '1.0',
    '--human_vpref_override', '1.0',
    '--shield_horizon_steps', str(SHIELD_HORIZON_STEPS),
    '--shield_safety_margin', str(SHIELD_SAFETY_MARGIN),
]
print('Running:', ' '.join(cmd))
print('=' * 80)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()
print()
print(f'Exited with code {p.returncode}')
if p.returncode != 0:
    raise SystemExit(p.returncode)


### 3.2 Run wide shield eval

Run this after the quick probe reports `GO`. This is still eval-only: no training, no checkpoint mutation.


In [ ]:
# Wide V38 decision eval: same C0/C1 setup, broader densities and 100 episodes.
BASE_CHECKPOINT = 'sncp_ppo_v34.pt'
OUTPUT_DIR = 'eval_v38_shield_full'
DENSITIES = [5, 10, 15, 20]
EVAL_EPISODES = 100
SHIELD_HORIZON_STEPS = 6
SHIELD_SAFETY_MARGIN = 0.0

import os, subprocess, sys

if not os.path.exists(BASE_CHECKPOINT):
    raise FileNotFoundError(
        f'{BASE_CHECKPOINT} not found. Upload/copy the locked v34-fixed-beta checkpoint '
        'to the repo root before running the v38 shield wide eval.'
    )

cmd = [
    sys.executable, '-u', 'scripts/run_v38_shield_probe.py',
    '--checkpoint', BASE_CHECKPOINT,
    '--output_dir', OUTPUT_DIR,
    '--densities', *[str(n) for n in DENSITIES],
    '--n_episodes', str(EVAL_EPISODES),
    '--seed', '100',
    '--robot_vpref', '1.0',
    '--human_vpref_override', '1.0',
    '--shield_horizon_steps', str(SHIELD_HORIZON_STEPS),
    '--shield_safety_margin', str(SHIELD_SAFETY_MARGIN),
]
print('Running:', ' '.join(cmd))
print('=' * 80)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()
print()
print(f'Exited with code {p.returncode}')
if p.returncode != 0:
    raise SystemExit(p.returncode)


### Resuming after a disconnect

V38 has no training state to resume. If Colab disconnects, rerun the setup cells, make sure
`sncp_ppo_v34.pt` is present, then rerun the quick or wide eval cell you need. Existing output folders can
be overwritten safely by changing `OUTPUT_DIR` or deleting the old folder.


## 4. V38 analysis

The runner writes `summary.json` and `report.md` directly in the output directory. This cell prefers the
wide eval (`eval_v38_shield_full/`) when present, otherwise it shows the quick probe (`eval_v38_shield_probe/`).


In [ ]:
import json, os
from IPython.display import Markdown, display

EVAL_OUT = 'eval_v38_shield_full' if os.path.exists('eval_v38_shield_full/summary.json') else 'eval_v38_shield_probe'
CHECKPOINT = 'sncp_ppo_v34.pt'

report = os.path.join(EVAL_OUT, 'report.md')
summary = os.path.join(EVAL_OUT, 'summary.json')
print('Reading:', EVAL_OUT)
if os.path.exists(report):
    with open(report, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
else:
    print(f'{report} not found. Run Section 3 first.')

if os.path.exists(summary):
    with open(summary, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print('Verdict:', data.get('verdict', {}).get('verdict'))


## 5. Visualize trajectories

Visualizers run in the v38 paper scenario (`paper_challenging`, robot 1.0 m/s). `CHECKPOINT` is set in
Section 4. The static trajectory uses `--action_shield` so it reflects C1 shielded behavior.


In [ ]:
# Single trajectory plot ? first successful episode out of 20 tries, with V38 shield enabled.
!python scripts/visualize_trajectory.py     --checkpoint {CHECKPOINT}     --output trajectory_plot.png     --num_humans 20     --scenario paper_challenging     --robot_vpref 1.0 --human_vpref_override 1.0 --human_goal_noise 0.0 --max_time 50     --action_shield --shield_horizon_steps 6 --shield_safety_margin 0.0     --seed 42

from IPython.display import Image, display
display(Image('trajectory_plot.png'))


In [ ]:
# Animated GIF for a single scenario (raw visualizer path; use static plot above for shielded C1).
!python scripts/visualize_trajectory_gif.py --checkpoint {CHECKPOINT} --num_humans 20 --scenario paper_challenging

from IPython.display import Image, display
import glob
gifs = sorted(glob.glob('*.gif'))
if gifs:
    print(f'Generated: {gifs}')
    display(Image(gifs[-1]))


## 6. Training curves

Plots the newest training CSV and shows the v22 diagnostics + artifact-verification reports.

In [ ]:
import json, os
from IPython.display import Markdown, display

EVAL_OUT = 'eval_v38_shield_full' if os.path.exists('eval_v38_shield_full/summary.json') else 'eval_v38_shield_probe'
if os.path.exists(f'{EVAL_OUT}/report.md'):
    with open(f'{EVAL_OUT}/report.md', 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
else:
    print(f'{EVAL_OUT}/report.md not found. Run Section 3 first.')

if os.path.exists(f'{EVAL_OUT}/summary.json'):
    with open(f'{EVAL_OUT}/summary.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(json.dumps(data.get('verdict', {}), indent=2))


### Inspect CSV in pandas (optional)

In [ ]:
import pandas as pd, glob
csv_files = sorted(glob.glob('logs/training_*.csv'))
if csv_files:
    df = pd.read_csv(csv_files[-1])
    print(f'Rows: {len(df)}')
    print(f'Columns: {list(df.columns)}')
    print('\nPhase distribution:')
    print(df['scenario'].value_counts().sort_index())
    print('\nHoldout success per eval (event points):')
    holdout_cols = [c for c in df.columns if c.startswith('holdout_') and c.endswith('_success')]
    if holdout_cols:
        hdf = df[holdout_cols].drop_duplicates()
        hdf.index = df.loc[hdf.index, 'episode']
        print(hdf.tail(10))

## 7. Persist results

With `USE_DRIVE=True`, artifacts may already be in Drive. Otherwise set `DOWNLOAD = True` to grab the
full wide-eval evidence bundle if it exists; the cell falls back to the quick probe bundle.


In [ ]:
from google.colab import files
import os, shutil

DOWNLOAD = False  # set True to trigger browser download dialogs
if DOWNLOAD:
    bundle_dir = 'eval_v38_shield_full' if os.path.isdir('eval_v38_shield_full') else 'eval_v38_shield_probe'
    if os.path.isdir(bundle_dir):
        archive = shutil.make_archive(f'{bundle_dir}_artifacts', 'zip', bundle_dir)
        files.download(archive)
    else:
        print('No V38 eval artifact directory found. Run Section 3 first.')


## 8. Notes & roadmap (current: v38 action shield)

`AGENTS.md` + the v38 plan are the source of truth.

### Story so far
- **v34-fixed-beta** remains the locked champion/base.
- **v36** full combined-levers run was negative/flat.
- **v37** paired probe was `NO-GO`; full v37 training is cancelled.
- **v38** is training-free: raw v34 action vs shielded v34 action.
- **Quick V38 probe** already reported `GO`; the current notebook includes the required wide eval.

### Decision rule
- Wide eval: N=5/10/15/20, 100 episodes per arm/density.
- `GO` requires high-N collision down by at least 3 pp, success not down more than 2 pp, timeout not up more than 2 pp.
- If wide eval is `GO`, V38 becomes the candidate shielded policy; if `NO-GO`, keep v34 unchanged and do not use the shield.
